In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import transformers
import torch

print("All libraries installed successfully ✅")

d:\customer-feedback-analyzer\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries installed successfully ✅


In [11]:
#STEP 1: LOAD DATASET

import pandas as pd

df = pd.read_csv("../data/reviews.csv")

# Keep only required columns
df = df[['Score', 'Text']]
df.columns = ['rating', 'text']

df.head()

,rating,text
0,5,I have bought several of the Vitality canned d...
1,1,Product arrived labeled as Jumbo Salted Peanut...
2,4,This is a confection that has been around a fe...
3,2,If you are looking for the secret ingredient i...
4,5,Great taffy at a great price. There was a wid...


In [12]:
#STEP 2: CREATE SENTIMENT LABELS (3-CLASS)

def get_sentiment(rating):
    if rating <= 2:
        return 0   # Negative
    elif rating == 3:
        return 1   # Neutral
    else:
        return 2   # Positive

df['label'] = df['rating'].apply(get_sentiment)

In [13]:
#STEP 3: VERIFY DATA

print(df['rating'].value_counts())
print(df['label'].value_counts())

rating
5    363122
4     80655
1     52268
3     42640
2     29769
Name: count, dtype: int64
label
2    443777
0     82037
1     42640
Name: count, dtype: int64


In [14]:
#STEP 4: TRAIN-TEST SPLIT

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

In [15]:
#STEP 5: KEEP REQUIRED COLUMNS

train_df = train_df[['text', 'label']]
test_df = test_df[['text', 'label']]

In [16]:
#STEP 6: TOKENIZER

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(
        example['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

In [17]:
#STEP 7: CONVERT TO DATASET (MEMORY SAFE)

from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df.sample(50000))
test_dataset = Dataset.from_pandas(test_df.sample(10000))

In [18]:
#STEP 8: TOKENIZE DATA

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 10000/10000 [00:01<00:00, 7340.46 examples/s]


In [19]:
#STEP 9: FIX LABEL COLUMN (VERY IMPORTANT)

train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

In [20]:
#STEP 10: FORMAT FOR TRAINING

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [21]:
#STEP 11: LOAD MODEL

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1472.48it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
#STEP 12: TRAINING SETUP

from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)

    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    logging_dir="./logs",
    eval_strategy="epoch",
    save_strategy="epoch"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [23]:
#STEP 13: TRAINER

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [24]:
#STEP 14: TRAIN MODEL

trainer.train()

d:\customer-feedback-analyzer\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.333529,0.314730,0.889500,0.887050
2,0.252356,0.355992,0.898400,0.894126


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.98it/s]
d:\customer-feedback-analyzer\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.33it/s]


TrainOutput(global_step=12500, training_loss=0.3109031585693359, metrics={'train_runtime': 29035.4685, 'train_samples_per_second': 3.444, 'train_steps_per_second': 0.431, 'total_flos': 3311744025600000.0, 'train_loss': 0.3109031585693359, 'epoch': 2.0})

In [25]:
#STEP 15: SAVE MODEL

trainer.save_model("models/sentiment_model")
tokenizer.save_pretrained("models/sentiment_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.13it/s]


('models/sentiment_model\\tokenizer_config.json',
 'models/sentiment_model\\tokenizer.json')

In [27]:
#STEP 16: TEST MODEL

import torch

text = "This product is amazing"

inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)

probs = torch.nn.functional.softmax(outputs.logits, dim=1)
pred = torch.argmax(probs).item()

labels = ["Negative 😡", "Neutral 😐", "Positive 😊"]

print(labels[pred])

Positive 😊
